<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Assessing and Fixing Agent Tools

## Overview
In this hands-on lab, you will identify and fix a poorly documented Python function that has been registered to Unity Catalog. You'll learn why proper documentation is critical for agent tools, fix the function description, verify it works via MCP, and test it in AI Playground.

## Learning Objectives
By the end of this lab, you will be able to:
- Identify why inadequate function descriptions break agent tool usage
- Fix a Python function's docstring following agent tool best practices
- Re-register an improved function to Unity Catalog using `DatabricksFunctionClient()`
- Discover and test tools via MCP (Model Context Protocol)
- Validate tools as agent tools using AI Playground

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Environment Setup

### A1. Install Dependencies
As part of the workspace setup, several Python libraries have been installed. To see the list of notebook-scoped libraries, please read this documentation ([AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#configure-environment-for-job-tasks) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#configure-environment-for-job-tasks) | [GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#configure-environment-for-job-tasks)). In particular, we installed:

1. `unitycatalog-ai[databricks]`: This package provides infrastructure and tooling for creating and managing UC functions (both SQL and Python UDFs) that can be used as tools by agents.

This demonstration uses AI Playground to test our functions, which provides a no-code interface for prototyping tool-calling agents. See the Unity Catalog Tool Integration documentation ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/unity-catalog-tool-integration) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/unity-catalog-tool-integration)) for more details for advanced framework integration.

In [0]:
!pip install unitycatalog-ai[databricks]==0.3.2
%restart_python

In [0]:
%run ./Includes/Classroom-Setup-3

### A2. Inspect the Airbnb Dataset

The classroom setup has loaded the Airbnb listings dataset into your schema. Let's take a look at the data our tools will operate on.

In [0]:
df = spark.read.table('sf_airbnb_listings')
display(df.limit(5))

### A3. Initialize the Databricks Function Client

**TODO:** Initialize the `DatabricksFunctionClient` for serverless compute. This client lets you register and execute UC functions programmatically.

In [0]:
## Import and initialize the DatabricksFunctionClient for serverless compute
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task A3: Initialize the Databricks Function Client
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()
```

</div>
</details>

## B. Examine a Poorly Documented Agent Tool

As part of the classroom setup, a Python function called `est_listing_value` has been registered to Unity Catalog. This function is intended to estimate the nightly value of an Airbnb listing based on its characteristics, but it was written with **inadequate documentation**.

Let's look at the function as it currently exists.

### B1. Review the Bad Function

The function defined below has already been registered to Unity Catalog. Notice the problems: vague parameter names, no useful docstring, and no business context. This function is already registered in UC -- here's what it looks like:


<div class="code-block-dark" data-language="python">
def est_listing_value(
    my_param: float,
    param2: float,
    param3: str
) -> str:
    """
    This is my calculation function.
<p>
    Args:
        my_param: a number
        param2: a number
        param3: a string
</p><p>
    Returns:
        str: result
    """
</p><p>
    base_value = my_param * 365 * 0.7  # assume 70% occupancy
    bedroom_factor = 1 + (param2 - 1) * 0.15
    if param3.lower() == 'entire home/apt':
        multiplier = 1.2
    elif param3.lower() == 'private room':
        multiplier = 0.8
    else:
        multiplier = 0.5
    estimated = base_value * bedroom_factor * multiplier
    return f"Estimated annual revenue: ${estimated:,.2f}"
</p>
</div>

<style id="prism-inline-css">
/* Prism Tomorrow Night theme - inlined */
code[class*="language-"],pre[class*="language-"]{color:#ccc;background:0 0;font-family:Consolas,Monaco,'Andale Mono','Ubuntu Mono',monospace;font-size:1em;text-align:left;white-space:pre;word-spacing:normal;word-break:normal;word-wrap:normal;line-height:1.5;tab-size:4;hyphens:none}pre[class*="language-"]{padding:1em;margin:.5em 0;overflow:auto}:not(pre)>code[class*="language-"],pre[class*="language-"]{background:#2d2d2d}:not(pre)>code[class*="language-"]{padding:.1em;border-radius:.3em;white-space:normal}.token.comment,.token.block-comment,.token.prolog,.token.doctype,.token.cdata{color:#999}.token.punctuation{color:#ccc}.token.tag,.token.attr-name,.token.namespace,.token.deleted{color:#e2777a}.token.function-name{color:#6196cc}.token.boolean,.token.number,.token.function{color:#f08d49}.token.property,.token.class-name,.token.constant,.token.symbol{color:#f8c555}.token.selector,.token.important,.token.atrule,.token.keyword,.token.builtin{color:#cc99cd}.token.string,.token.char,.token.attr-value,.token.regex,.token.variable{color:#7ec699}.token.operator,.token.entity,.token.url{color:#67cdcc}.token.important,.token.bold{font-weight:700}.token.italic{font-style:italic}.token.entity{cursor:help}.token.inserted{color:green}
</style>

<script>
// Minimal Prism core + Python highlighting
(function(){
function escapeHtml(text) {
    var div = document.createElement('div');
    div.textContent = text;
    return div.innerHTML;
}

var Prism = window.Prism = {
    languages: {},
    highlight: function(text, grammar) {
        return this.tokenize(text, grammar).map(function(token) {
            if (typeof token === 'string') return escapeHtml(token);
            var type = token.type, content = token.content;
            return '<span class="token ' + type + '">' + escapeHtml(content) + '</span>';
        }).join('');
    },
    tokenize: function(text, grammar) {
        var tokens = [text];
        for (var key in grammar) {
            var pattern = grammar[key];
            for (var i = 0; i < tokens.length; i++) {
                if (typeof tokens[i] !== 'string') continue;
                var match, matches = [];
                pattern.lastIndex = 0;
                while ((match = pattern.exec(tokens[i])) !== null) {
                    matches.push({index: match.index, length: match[0].length, value: match[0]});
                }
                if (!matches.length) continue;
                var newTokens = [];
                var lastIndex = 0;
                matches.forEach(function(m) {
                    if (m.index > lastIndex) newTokens.push(tokens[i].substring(lastIndex, m.index));
                    newTokens.push({type: key, content: m.value});
                    lastIndex = m.index + m.length;
                });
                if (lastIndex < tokens[i].length) newTokens.push(tokens[i].substring(lastIndex));
                tokens.splice.apply(tokens, [i, 1].concat(newTokens));
            }
        }
        return tokens;
    },
    highlightElement: function(element) {
        var code = element.textContent;
        var grammar = this.languages.python;
        element.innerHTML = this.highlight(code, grammar);
    }
};

Prism.languages.python = {
    'comment': /#.*/g,
    'string': /("""[\s\S]*?"""|'''[\s\S]*?'''|"[^"]*"|'[^']*')/g,
    'keyword': /\b(import|from|def|class|return|if|elif|else|for|while|try|except|finally|with|as|pass|break|continue|yield|lambda|async|await|None|True|False)\b/g,
    'function': /\b\w+(?=\()/g,
    'number': /\b\d+\.?\d*\b/g,
    'operator': /[-+*/%=<>!&|^~]+/g,
    'punctuation': /[{}[\];(),.]/g
};

document.querySelectorAll('.code-block-dark').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var id = 'code-dark-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML =
        '<div style="position:relative;margin:16px 0;">' +
            '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#555;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;">Copy</button>' +
            '<pre style="background:#2d2d2d;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #444;"><code id="' + id + '" class="language-python"></code></pre>' +
        '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = '✓ Copied!';
        var btn = this;
        setTimeout(function(){ btn.textContent='Copy'; },2000);
    };
});
})();
</script>


### B2. Test in AI Playground

Navigate to **AI Playground** and attach the tool `est_listing_value`. Select the GPT-5.4 model

Try asking:

<div class="code-block-light" data-language="text">
I have a 2-bedroom entire-home Airbnb listing in San Francisco priced at $250. What could I expect to make from it per year?
</div>

Notice how the agent struggles with the vague parameter names and description. It may pass parameters incorrectly or misinterpret the function's purpose entirely.


<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

## C. Fix the Function Description

The function logic is correct -- the problem is entirely in the documentation. An agent reads the function name, docstring, and parameter descriptions to decide:
1. **When** to call the function
2. **What** values to pass for each parameter
3. **How** to interpret the result

Without clear documentation, the agent is guessing.

### C1. Rewrite the Function with Proper Documentation

**TODO:** Redefine the function with:
- Descriptive parameter names (not `my_param` and `param2`)
- A clear docstring explaining the function's purpose
- Detailed parameter descriptions with types, units, and examples
- A clear return value description

In [0]:
## Redefine the function with an improved docstring
def est_listing_value(
    nightly_price_usd:float,
    bedroom_count:int,
    room_type:str
) -> str:
    """
    Estimate the annual revenue of an Airbnb listing based on its nightly price, # of bedrooms, and listing description.

    Useful for questions like: 'I have a x bedroom listing priced at $xxx. What could I expect to make from it per year?',
    'What's my listing value per year?'

    Args:
        nightly_price_usd: Nightly listing price in USD.
        bedroom_count: Number of bedrooms in the listing.
        room_type: Room type of the listing.

    Returns:
        Estimated annual revenue in USD.
    """
    base_value = nightly_price_usd * 365 * 0.7  # assume 70% occupancy
    bedroom_factor = 1 + (bedroom_count - 1) * 0.15
    if room_type.lower() == 'entire home/apt':
        multiplier = 1.2
    elif room_type.lower() == 'private room':
        multiplier = 0.8
    else:
        multiplier = 0.5
    estimated = base_value * bedroom_factor * multiplier
    return f"Estimated annual revenue: ${estimated:,.2f}"


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C1: Rewrite the Function
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
def est_listing_value(
    nightly_price_usd: float,
    num_bedrooms: float,
    room_type: str
) -> str:
    """
    Estimate the annual revenue for a San Francisco Airbnb listing based on
    its nightly price, number of bedrooms, and room type. Assumes 70%
    occupancy year-round.

    Use this tool when a user asks how much a listing could earn per year.

    Args:
        nightly_price_usd: The nightly price of the listing in US dollars (e.g. 250.0).
        num_bedrooms: The number of bedrooms in the listing (e.g. 1, 2, 3). A
            studio should be passed as 1. Each additional bedroom beyond
            the first increases the estimate by 15%.
        room_type: The Airbnb room type. Must be exactly one of:
            "Entire home/apt", "Private room", or "Shared room".

    Returns:
        A string describing the estimated annual revenue in USD.
    """
    base_value = nightly_price_usd * 365 * 0.7  # assume 70% occupancy
    bedroom_factor = 1 + (num_bedrooms - 1) * 0.15
    if room_type.lower() == 'entire home/apt':
        multiplier = 1.2
    elif room_type.lower() == 'private room':
        multiplier = 0.8
    else:
        multiplier = 0.5
    estimated = base_value * bedroom_factor * multiplier
    return f"Estimated annual revenue: ${estimated:,.2f}"
```

</div>
</details>

### C2. Test the Improved Function Locally

Before re-registering, verify the function works correctly at the notebook level.

In [0]:
result = est_listing_value(nightly_price_usd=250.0, bedroom_count=2, room_type="entire home/apt")
print(result)

### C3. Re-register the Function to Unity Catalog

**TODO:** Use `DatabricksFunctionClient` to re-register the improved function with `replace=True`.

In [0]:
function_info = client.create_python_function(
    func=est_listing_value,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C3: Re-register the Function
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
function_info = client.create_python_function(
    func=est_listing_value,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)
```

</div>
</details>

## D. Verify the Improved Tool via MCP

In the demo, you saw how MCP allows agents to discover and call UC functions. Let's use MCP to verify that our re-registered function has the improved description and works correctly.

### D1. Discover the Tool

**TODO:** Initialize the MCP client and list available tools. Verify that `est_listing_value` appears with the improved description.

In [0]:
import nest_asyncio
nest_asyncio.apply()

# TODO
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description[:150]}")
    print()


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D1: Discover the Tool via MCP
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description[:150]}")
    print()
```

</div>
</details>

### D2. Call the Tool via MCP

**TODO:** Execute `est_listing_value` through MCP to verify it works end-to-end.

In [0]:
result = mcp_client.call_tool(f"{catalog_name}__{schema_name}__est_listing_value",  {"nightly_price_usd": 250.0, "bedroom_count": 2, "room_type": "entire home/apt"})

print(f"MCP Result: {result}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D2: Call the Tool via MCP
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
result = mcp_client.call_tool(f"{catalog_name}__{schema_name}__est_listing_value", 
{
    "nightly_price_usd": 250.0, 
    "num_bedrooms":2, 
    "room_type":"entire home/apt"
    }
 )

print(f"MCP Result: {result}")
```

</div>
</details>

## E. Validate with AI Playground

Now let's test the improved function in AI Playground to see the difference that proper documentation makes.

### E1. Attach and Test

1. Navigate to **Playground** from your Databricks workspace
2. Select a tool-capable model (e.g., GPT-5.1 or Claude)
3. Click **Tools** and attach `est_listing_value` from your catalog and schema
4. Try the same query that failed before:

<div class="code-block-light" data-language="text">
I have a 2-bedroom entire-home Airbnb listing in San Francisco priced at $250. What could I expect to make from it per year?
</div>
Notice how the agent now correctly identifies the parameters and passes them appropriately. The improved documentation gives the agent the context it needs to use the tool effectively.


<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>


## Conclusion

In this lab you learned that **documentation is the interface** between an agent and its tools. The same function logic can be useless or powerful depending on how well it's documented. You also verified your tools work through MCP -- the same protocol that agents use in production to discover and call tools.

Key takeaways:
- Poor parameter names and docstrings make tools unusable by agents
- Good documentation includes: clear names, detailed descriptions with examples, and business context
- MCP provides a standard way for agents to discover and call UC functions
- Always test tools via MCP and AI Playground before integrating with agent code

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>